## Setup

In [0]:
pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

## Single Retriever

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

index_name = "global_fs.worldbank.chapter_index"
vector_search_endpoint_name = "one-env-shared-endpoint-13"

vs_index = vsc.get_index(endpoint_name=vector_search_endpoint_name, index_name=index_name)

results = vs_index.similarity_search(
  query_text="Greek myths",
  columns=all_columns,
  filters={"id NOT": ("13770", "88231")},
  num_results=2)

In [0]:
from typing import List, Dict, Optional, Any
from databricks.vector_search.client import VectorSearchClient


class VectorSearchWrapper:
    def __init__(
        self,
        endpoint_name: str,
        index_name: str,
        default_columns: List[str],
        default_filters: Optional[Dict[str, tuple]] = {},
        default_num_results: int = 10,
    ):
        """
        Initialize the VectorSearchWrapper.

        :param endpoint_name: The name of the vector search endpoint.
        :param index_name: The name of the index to use.
        :param default_query_text: Default query text if none provided.
        :param default_columns: Default columns to return in results.
        :param default_filters: Default filters to apply.
        :param default_num_results: Default number of results to return.
        """
        self.vsc = VectorSearchClient()
        self.endpoint_name = endpoint_name
        self.index_name = index_name
        self.index = self.vsc.get_index(
            endpoint_name=self.endpoint_name, index_name=self.index_name
        )

        self.default_columns = default_columns
        self.default_filters = default_filters
        self.default_num_results = default_num_results

    def search(
        self,
        query_text,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ):
        """
        Perform a similarity search against the configured index.

        :param query_text: The text query to search for (defaults to self.default_query_text).
        :param columns: List of columns to return in results (defaults to self.default_columns).
        :param filters: Optional filters to apply (defaults to self.default_filters).
        :param num_results: Number of results to return (defaults to self.default_num_results).
        :return: Search results from the vector index.
        """
        return self.index.similarity_search(
            query_text=query_text or self.default_query_text,
            columns=columns or self.default_columns,
            filters=filters or self.default_filters,
            num_results=num_results or self.default_num_results,
        )

In [0]:
# Example usage

index_name = "global_fs.worldbank.chapter_index"
vector_search_endpoint_name = "one-env-shared-endpoint-13"

client = VectorSearchWrapper(
    endpoint_name=vector_search_endpoint_name,
    index_name=index_name,
    default_columns=["chapter", "chapter_text"],
    default_filters={},
    default_num_results=2,
)

# Uses defaults
results = client.search("hello")
print(results)

## Multi Retriever

In [0]:
from typing import List, Dict, Optional, Any
from databricks_langchain import ChatDatabricks
import ast


class MultiRetrieverOrchestrator:
    def __init__(
        self,
        retriever_configs: List[Dict[str, Any]],
        llm_endpoint: str = "databricks-gpt-oss-120b",
    ):
        """
        Initialize with a list of retriever configurations.

        Each config should include:
            - endpoint_name
            - index_name
            - (optional) default_query_text
            - (optional) default_columns
            - (optional) default_filters
            - (optional) default_num_results

        :param llm_endpoint: The LLM endpoint used to generate queries.
        """
        self.retrievers = [
            VectorSearchWrapper(**config) for config in retriever_configs
        ]
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)

    def generate_queries(self, query_text: str) -> List[str]:
        """
        Use the LLM to generate a list of queries, one per retriever.

        :param query_text: The original query from the user.
        :return: List of LLM-generated queries with length equal to number of retrievers.
        """
        num_queries = len(self.retrievers)

        response = self.llm.invoke(
            f"Generate {num_queries} variations of this query for vector search: '{query_text}'. Return the generated queries in a list. For example ['how are you?', 'how do you do?']"
        )

        text_items = [item['text'] for item in response.content if item.get('type') == 'text']
        if not text_items:
            raise ValueError("LLM response does not contain any items with type='text'")

        # Assume the first text item contains the list string
        text_str = text_items[0]

        # Convert string representation of list to Python list
        import ast
        try:
            queries = ast.literal_eval(text_str)
        except Exception as e:
            raise ValueError(f"Failed to parse LLM text as list: {e}")

        return queries

    def search_all(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Run all retrievers in parallel using LLM-generated queries.

        :param query_text: Original query that will be fed to LLM.
        :return: Dict mapping retriever index -> results.
        """
        llm_queries = self.generate_queries(query_text)

        results = {}
        with ThreadPoolExecutor(max_workers=len(self.retrievers)) as executor:
            futures = {
                executor.submit(r.search, llm_query, columns, filters, num_results): i
                for i, (r, llm_query) in enumerate(
                    zip(self.retrievers, llm_queries), start=1
                )
            }
            for future in as_completed(futures):
                key = f"retriever_{futures[future]}"
                try:
                    results[key] = future.result()
                except Exception as e:
                    results[key] = f"Error: {e}"

        return results

In [0]:
retriever_configs = [
    {
        "endpoint_name": "one-env-shared-endpoint-13",
        "index_name": "global_fs.worldbank.chapter_index",
        "default_columns": ["chapter", "chapter_text"],
        "default_filters": {},
        "default_num_results": 2
    },
    {
        "endpoint_name": "one-env-shared-endpoint-13",
        "index_name": "global_fs.worldbank.chapter_index",
        "default_columns": ["chapter", "chapter_text"],
        "default_filters": {},
        "default_num_results": 5
    }
]


In [0]:
orchestrator = MultiRetrieverOrchestrator(retriever_configs)

In [0]:
import mlflow
mlflow.langchain.autolog()

results = orchestrator.search_all(
    query_text="hello"
)

In [0]:
results